# Practice - 11 - Query plans, skew, shuffle tuning, and table layout

Prerequisites / execution context: Run 00. Small bounded dataset. Plan inspection works broadly; detailed Spark UI and cluster setting exercises require compatible classic compute. No benchmark claims from tiny data.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R18: PySpark overview](https://docs.databricks.com/aws/en/pyspark/); [R52: Spark UI diagnosis](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/); [R53: Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page); [R54: Spark memory issues](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/spark-memory-issues); [R55: Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe); [R56: Liquid clustering](https://docs.databricks.com/aws/en/tables/clustering); [R57: Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Inspect skew and a broadcast plan

Create 100000 synthetic rows: k=0 for id<90000, otherwise id%100; amount=1. Name fact. Create dim for keys 0..99 with a string label. Inspect key frequencies, broadcast-join dim and explain the plan. Name joined.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert joined.count() == 100000
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIEJ1aWxkIGEgc2tld2VkIGRpc3RyaWJ1dGlvbgpNb3N0IGV2ZW50cyBoYXZlIHRoZSBzYW1lIGtleS4gSW5zcGVjdCB0aGUgcGxhbiBhbmQgcGVyLWtleSBkaXN0cmlidXRpb24gYmVmb3JlIGNoYW5naW5nIHNldHRpbmdzLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmZhY3Q9KHNwYXJrLnJhbmdlKDEwMDAwMCkud2l0aENvbHVtbigmcXVvdDtrJnF1b3Q7LEYud2hlbihGLmNvbCgmcXVvdDtpZCZxdW90OykmbHQ7OTAwMDAsRi5saXQoMCkpLm90aGVyd2lzZShGLmNvbCgmcXVvdDtpZCZxdW90OyklMTAwKSkKICAgICAgLndpdGhDb2x1bW4oJnF1b3Q7YW1vdW50JnF1b3Q7LEYubGl0KDEpKSkKZGltPXNwYXJrLnJhbmdlKDEwMCkuc2VsZWN0KEYuY29sKCZxdW90O2lkJnF1b3Q7KS5hbGlhcygmcXVvdDtrJnF1b3Q7KSxGLmNvbmNhdChGLmxpdCgmcXVvdDtncm91cF8mcXVvdDspLEYuY29sKCZxdW90O2lkJnF1b3Q7KSkuYWxpYXMoJnF1b3Q7bGFiZWwmcXVvdDspKQpmYWN0Lmdyb3VwQnkoJnF1b3Q7ayZxdW90OykuY291bnQoKS5vcmRlckJ5KEYuZGVzYygmcXVvdDtjb3VudCZxdW90OykpLnNob3coNSkKam9pbmVkPWZhY3Quam9pbihGLmJyb2FkY2FzdChkaW0pLCZxdW90O2smcXVvdDspCmpvaW5lZC5leHBsYWluKCZxdW90O2Zvcm1hdHRlZCZxdW90OykKYXNzZXJ0IGpvaW5lZC5jb3VudCgpPT0xMDAwMDAKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Coding task 2: Salt a decomposable aggregation

Compute plain SUM(amount) by k. Then compute salted with 16 buckets derived from id, partial sums by k,salt and final sums by k. Use differences in BOTH directions to prove equal results.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert plain.exceptAll(salted).count() == 0
assert salted.exceptAll(plain).count() == 0
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIFZlcmlmeSBhIHNhbHRlZCBhZ2dyZWdhdGlvbiBwcmVzZXJ2ZXMgdGhlIGFuc3dlcgpQYXJ0aWFsIHN1bXMgZGlzdHJpYnV0ZSBhIGhvdCBrZXk7IGNvbWJpbmUgcGFydGlhbCBzdW1zIGJ5IHRoZSBvcmlnaW5hbCBrZXkuIFRoaXMgYXBwbGllcyB0byBhIGRlY29tcG9zYWJsZSBTVU0uIEF2ZXJhZ2VzIG5lZWQgc3VtIGFuZCBjb3VudDsgZXhhY3QgZGlzdGluY3QgbmVlZHMgZGlmZmVyZW50IHJlYXNvbmluZy4gQVFFIHNrZXctam9pbiBoYW5kbGluZyBpcyBub3QgYSB1bml2ZXJzYWwgZml4IGZvciBza2V3ZWQgYWdncmVnYXRpb24uIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9vcHRpbWl6YXRpb25zL3NwYXJrLXVpLWd1aWRlL2xvbmctc3Bhcmstc3RhZ2UtcGFnZSIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI1MzogU2tldyBhbmQgc3BpbGw8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL29wdGltaXphdGlvbnMvYXFlIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjU1OiBBZGFwdGl2ZSBxdWVyeSBleGVjdXRpb248L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnBsYWluPWZhY3QuZ3JvdXBCeSgmcXVvdDtrJnF1b3Q7KS5hZ2coRi5zdW0oJnF1b3Q7YW1vdW50JnF1b3Q7KS5hbGlhcygmcXVvdDt0b3RhbCZxdW90OykpCnNhbHRlZD0oZmFjdC53aXRoQ29sdW1uKCZxdW90O3NhbHQmcXVvdDssKEYuY29sKCZxdW90O2lkJnF1b3Q7KSUxNikuY2FzdCgmcXVvdDtpbnQmcXVvdDspKQogLmdyb3VwQnkoJnF1b3Q7ayZxdW90OywmcXVvdDtzYWx0JnF1b3Q7KS5hZ2coRi5zdW0oJnF1b3Q7YW1vdW50JnF1b3Q7KS5hbGlhcygmcXVvdDtwYXJ0aWFsJnF1b3Q7KSkKIC5ncm91cEJ5KCZxdW90O2smcXVvdDspLmFnZyhGLnN1bSgmcXVvdDtwYXJ0aWFsJnF1b3Q7KS5hbGlhcygmcXVvdDt0b3RhbCZxdW90OykpKQphc3NlcnQgcGxhaW4uZXhjZXB0QWxsKHNhbHRlZCkuY291bnQoKT09MAphc3NlcnQgc2FsdGVkLmV4Y2VwdEFsbChwbGFpbikuY291bnQoKT09MApzYWx0ZWQuZXhwbGFpbigmcXVvdDtmb3JtYXR0ZWQmcXVvdDspCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 3: Optional: measure partition changes

On compatible classic compute, compare shuffle partitions 8 and 32. Record elapsed time of the same aggregation; restore the original setting in finally. Do not claim a tiny-data production benchmark.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
RUN_CLASSIC_TUNING = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIE1lYXN1cmUgb25lIGNoYW5nZSBhdCBhIHRpbWUKT24gY29tcGF0aWJsZSBjbGFzc2ljIGNvbXB1dGU6IGNhcHR1cmUgYmFzZWxpbmUgcnVudGltZSwgaW5wdXQgc2l6ZSwgc2h1ZmZsZSByZWFkL3dyaXRlLCBzcGlsbCwgYW5kIG1heC9tZWRpYW4gdGFzayBkdXJhdGlvbi4gQ29tcGFyZSBzaHVmZmxlIHBhcnRpdGlvbiBzZXR0aW5nczsgcmVzdG9yZSB0aGUgb3JpZ2luYWwgc2V0dGluZyBhZnRlcndhcmQuIE1vcmUgcGFydGl0aW9ucyByZWR1Y2Ugd29yayBwZXIgdGFzayBidXQgYWRkIG92ZXJoZWFkLiBNZW1vcnkgYW5kIGRlZmF1bHQgcGFyYWxsZWxpc20gYXJlIG5vdCBpbnRlcmNoYW5nZWFibGUgd2l0aCBTUUwgc2h1ZmZsZSBwYXJ0aXRpb25zLiBPbiBzZXJ2ZXJsZXNzLCBtYW55IHNldHRpbmdzIGNhbm5vdCBiZSBjb25maWd1cmVkIGFuZCBTcGFyayBVSSBhY2Nlc3MgZGlmZmVyczsgdXNlIGF2YWlsYWJsZSBxdWVyeSBwcm9maWxlcy4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2NvbXB1dGUvc2VydmVybGVzcy9saW1pdGF0aW9ucyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIwNDogU2VydmVybGVzcyBjb21wdXRlIGxpbWl0YXRpb25zPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9vcHRpbWl6YXRpb25zL3NwYXJrLXVpLWd1aWRlLyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI1MjogU3BhcmsgVUkgZGlhZ25vc2lzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9vcHRpbWl6YXRpb25zL3NwYXJrLXVpLWd1aWRlL3NwYXJrLW1lbW9yeS1pc3N1ZXMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNTQ6IFNwYXJrIG1lbW9yeSBpc3N1ZXM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL29wdGltaXphdGlvbnMvYXFlIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjU1OiBBZGFwdGl2ZSBxdWVyeSBleGVjdXRpb248L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlJVTl9DTEFTU0lDX1RVTklORz1GYWxzZQppZiBSVU5fQ0xBU1NJQ19UVU5JTkc6CiAgICBpbXBvcnQgdGltZQogICAga2V5PSZxdW90O3NwYXJrLnNxbC5zaHVmZmxlLnBhcnRpdGlvbnMmcXVvdDsKICAgIG9sZD1zcGFyay5jb25mLmdldChrZXkpCiAgICBtZWFzdXJlbWVudHM9W10KICAgIHRyeToKICAgICAgICBmb3Igc2V0dGluZyBpbiAoJnF1b3Q7OCZxdW90OywmcXVvdDszMiZxdW90Oyk6CiAgICAgICAgICAgIHNwYXJrLmNvbmYuc2V0KGtleSxzZXR0aW5nKQogICAgICAgICAgICBzdGFydD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZhY3QuZ3JvdXBCeSgmcXVvdDtrJnF1b3Q7KS5zdW0oJnF1b3Q7YW1vdW50JnF1b3Q7KS5jb2xsZWN0KCkgICMgYm91bmRlZCAxMDAtZ3JvdXAgcmVzdWx0CiAgICAgICAgICAgIG1lYXN1cmVtZW50cy5hcHBlbmQoKHNldHRpbmcsdGltZS5wZXJmX2NvdW50ZXIoKS1zdGFydCkpCiAgICBmaW5hbGx5OgogICAgICAgIHNwYXJrLmNvbmYuc2V0KGtleSxvbGQpCiAgICBwcmludChtZWFzdXJlbWVudHMpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 4: Optional: create a liquid-clustered sandbox

Create SILVER.cluster_demo(id BIGINT,k BIGINT) CLUSTER BY(k), append fact IDs/keys, optimize and inspect details. Keep disabled unless the runtime/table feature is supported.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
RUN_CLUSTERING = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA0IGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDQuIExpcXVpZCBjbHVzdGVyaW5nIHNhbmRib3gKVGhpcyBvcHRpb25hbCBvcGVyYXRpb24gcmVxdWlyZXMgc3VwcG9ydGVkIHRhYmxlL3J1bnRpbWUgZmVhdHVyZXM7IHNraXAgaWYgdW5hdmFpbGFibGUuIFRpbnkgZGF0YSBjYW5ub3QgZGVtb25zdHJhdGUgcHJ1bmluZyBiZW5lZml0cy4gRm9yIGEgbWVhbmluZ2Z1bCBjb21wYXJpc29uIHVzZSByZXByZXNlbnRhdGl2ZSBwcm9kdWN0aW9uLXNoYXBlZCBkYXRhIGluIGEgc2VwYXJhdGUgbGFiLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vdGFibGVzL2NsdXN0ZXJpbmciIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNTY6IExpcXVpZCBjbHVzdGVyaW5nPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA0IHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlJVTl9DTFVTVEVSSU5HPUZhbHNlCmlmIFJVTl9DTFVTVEVSSU5HOgogICAgVD1mJnF1b3Q7e1NJTFZFUn0uY2x1c3Rlcl9kZW1vJnF1b3Q7CiAgICBzcGFyay5zcWwoZiZxdW90O0NSRUFURSBPUiBSRVBMQUNFIFRBQkxFIHtUfSAoaWQgQklHSU5ULGsgQklHSU5UKSBVU0lORyBERUxUQSBDTFVTVEVSIEJZIChrKSZxdW90OykKICAgIGZhY3Quc2VsZWN0KCZxdW90O2lkJnF1b3Q7LCZxdW90O2smcXVvdDspLndyaXRlLm1vZGUoJnF1b3Q7YXBwZW5kJnF1b3Q7KS5zYXZlQXNUYWJsZShUKQogICAgc3Bhcmsuc3FsKGYmcXVvdDtPUFRJTUlaRSB7VH0mcXVvdDspCiAgICBzcGFyay5zcWwoZiZxdW90O0RFU0NSSUJFIERFVEFJTCB7VH0mcXVvdDspLnNob3codHJ1bmNhdGU9RmFsc2UpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 1

Diagnose: one huge slow task; widespread spill; driver OOM after collect; failure before the first Spark task. Distinguish predictive optimization from fixing an incorrect join.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgRGlhZ25vc3RpYyBjYXNlcwpBIGZldyBzbG93IGh1Z2UgdGFza3M6IGluc3BlY3Qgc2tldy4gTWFueSBsYXJnZSBzcGlsbGluZyB0YXNrczogaW5zcGVjdCBwYXJ0aXRpb24gc2l6aW5nIGFuZCBtZW1vcnkgcGVyIGNvcmUuIERyaXZlciBPT00gYWZ0ZXIgY29sbGVjdDogcmVkdWNlIGRyaXZlciBjb2xsZWN0aW9uLiBGYWlsdXJlIGJlZm9yZSBmaXJzdCB0YXNrOiBpbnNwZWN0IGNvbXB1dGUgZXZlbnRzLCBwZXJtaXNzaW9ucywgbmV0d29ya2luZywgb3IgbGlicmFyeSBlbnZpcm9ubWVudC4gUHJlZGljdGl2ZSBvcHRpbWl6YXRpb24gYXV0b21hdGVzIG1haW50ZW5hbmNlIGZvciBlbGlnaWJsZSBVQyBtYW5hZ2VkIHRhYmxlczsgaXQgZG9lcyBub3QgcmVwYWlyIGFuIGluY29ycmVjdCBqb2luLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vb3B0aW1pemF0aW9ucy9zcGFyay11aS1ndWlkZS8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNTI6IFNwYXJrIFVJIGRpYWdub3NpczwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vb3B0aW1pemF0aW9ucy9zcGFyay11aS1ndWlkZS9zcGFyay1tZW1vcnktaXNzdWVzIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjU0OiBTcGFyayBtZW1vcnkgaXNzdWVzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9vcHRpbWl6YXRpb25zL3ByZWRpY3RpdmUtb3B0aW1pemF0aW9uIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjU3OiBQcmVkaWN0aXZlIG9wdGltaXphdGlvbjwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')
